# Gold Layer — Build Dimension Tables
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | `<your-catalog>.silver.*` (already cleaned, conformed) |
| **Target** | `<your-catalog>.gold.dim_*` |
| **Write strategy** | Full overwrite — Silver already accumulates SCD history; Gold just re-publishes it |

### Dimensions built in this notebook
| Dimension | SCD | Surrogate Key |
|---|---|---|
| `dim_customer` | SCD2 | `customer_sk` (reused from Silver) |
| `dim_product` | SCD2 | `product_sk` (reused from Silver) |
| `dim_date` | Static, generated | `date_key` |
| `dim_address` | SCD1 | `address_sk` (new — generated here) |
| `dim_payment_method` | SCD1 (lookup) | `payment_method_sk` (new — generated here) |

> **Why a full overwrite, not `MERGE INTO`?** The SCD1/SCD2 decision-making
> already happened in Silver — `silver.orders` has a real `MERGE INTO` for
> SCD1, and `silver.customers`/`silver.products` will get the SCD2
> `MERGE INTO` in the Day 12 incremental notebook. By the time Gold reads
> Silver, the rows are already correctly versioned. A `MERGE` here would
> just match `customer_sk` to itself — no actual decision left to make.
> At GlobalMart's scale (~20K customers, 500 products), a full overwrite
> is simpler and equally correct.
>
> **Code below uses the literal `gbmart` catalog** — this is GlobalMart's
> real Gold-layer run; your own catalog will be named differently.

## Step 1 — Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# CATALOG = "gbmart"
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

## Step 2 — `dim_customer` (SCD2)

| Column | Why it's here |
|---|---|
| `customer_sk` | Join target for `fact_sales`; decouples the fact from the natural key and is what makes SCD2 versioning possible |
| `customer_id` | Business key — traces back to the source system, same value across all versions of one customer |
| `full_name`, `email`, `phone_number` | What the business actually looks at; `email`/`phone_number` are also exactly the fields that trigger a new SCD2 version when they change |
| `is_current` | Lets new `fact_sales` rows find "the current version" without date-range logic |
| `effective_start_date`, `effective_end_date` | What makes point-in-time history possible |

In [0]:
dim_customer_df = spark.table("gbmart.silver.customers").select(
    "customer_sk", "customer_id", "full_name", "email", "phone_number",
    "is_current", "effective_start_date", "effective_end_date"
)

dim_customer_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_customer")
print(f"dim_customer rows: {spark.table('gbmart.gold.dim_customer').count():,}")

## Step 3 — `dim_product` (SCD2)

| Column | Why it's here |
|---|---|
| `product_sk` | Join target for `fact_sales`; supports SCD2 versioning |
| `product_id` | Business key — same product across all its price-version rows |
| `product_name`, `category`, `sub_category` | What reports group/filter by — "revenue by category" needs these |
| `discounted_price_inr` | The attribute that actually changes and is the reason this dimension is SCD2 — price history is the textbook use case |
| `is_current`, `effective_start_date`, `effective_end_date` | SCD2 control columns, same role as in `dim_customer` |

In [0]:
dim_product_df = spark.table("gbmart.silver.products").select(
    "product_sk", "product_id", "product_name", "category", "sub_category",
    "discounted_price_inr", "is_current", "effective_start_date", "effective_end_date"
)

dim_product_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_product")
print(f"dim_product rows: {spark.table('gbmart.gold.dim_product').count():,}")

## Step 4 — `dim_date` (static, generated — not from Silver)

Date range is taken from actual data bounds (earliest order, latest delivery),
not an arbitrary hardcoded range — covers exactly what `fact_sales` needs,
nothing more.

| Column | Why it's here |
|---|---|
| `date_key` | Integer surrogate key in `yyyyMMdd` form — standard Kimball convention, faster to join/partition on than a `DATE` type |
| `date` | The actual date, for display and date-math in BI tools |
| `day_of_week`, `month`, `quarter`, `year` | Pre-computed time hierarchy — avoids every query recomputing `QUARTER(date)` itself |
| `is_weekend` | Common business flag — "do we sell more on weekends?" without date-math in every query |

In [0]:
date_bounds = spark.table("gbmart.silver.orders").select(
    min("order_date").alias("min_date"),
    max("actual_delivery_date").alias("max_date")
).collect()[0]

min_date, max_date = date_bounds["min_date"], date_bounds["max_date"]
print(f"Date range needed: {min_date} to {max_date}")

date_range_df = spark.sql(f"""
    SELECT explode(sequence(to_date('{min_date}'), to_date('{max_date}'), interval 1 day)) AS date
""")

dim_date_df = date_range_df.select(
    date_format(col("date"), "yyyyMMdd").cast("int").alias("date_key"),
    col("date"),
    date_format(col("date"), "EEEE").alias("day_of_week"),
    month(col("date")).alias("month"),
    quarter(col("date")).alias("quarter"),
    year(col("date")).alias("year"),
    (dayofweek(col("date")).isin([1, 7])).alias("is_weekend")
)

dim_date_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_date")
print(f"dim_date rows: {spark.table('gbmart.gold.dim_date').count():,}")

## Step 5 — `dim_address` (SCD1 — surrogate key generated here, since `silver.address` doesn't have one)

| Column | Why it's here |
|---|---|
| `address_sk` | Even an SCD1 dimension gets a surrogate key — keeps the join pattern uniform across every dimension, so `fact_sales` doesn't need special-case logic per dimension |
| `address_id` | Business key, traces back to source |
| `customer_id` | Lets you filter "this customer's addresses" without going through the fact table |
| `city`, `state`, `pincode` | The entire reason this dimension exists — "revenue by city/state" reporting |
| `address_type` | Distinguishes Billing vs Shipping when it matters for a fact row |

In [0]:
dim_address_df = spark.table("gbmart.silver.address").select(
    sha2(col("address_id"), 256).alias("address_sk"),
    "address_id", "customer_id", "city", "state", "pincode", "address_type"
)

dim_address_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_address")
print(f"dim_address rows: {spark.table('gbmart.gold.dim_address').count():,}")

## Step 6 — `dim_payment_method` (lookup — surrogate key generated here)

| Column | Why it's here |
|---|---|
| `payment_method_sk` | Same uniformity reason as `dim_address` — every dimension joins to the fact the same way |
| `payment_method_id` | Business key (`PM-001`), traces back to source |
| `method_name` | The human-readable value ("Credit Card") — `PM-001` means nothing to a business user on its own |

In [0]:
dim_payment_method_df = spark.table("gbmart.silver.payment_methods").select(
    sha2(col("payment_method_id"), 256).alias("payment_method_sk"),
    "payment_method_id", "method_name"
)

dim_payment_method_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_payment_method")
print(f"dim_payment_method rows: {spark.table('gbmart.gold.dim_payment_method').count():,}")

## Step 7 — `dim_orders` (SCD1 — surrogate key generated here)

| Column | Why it's here |
|---|---|
| `order_sk` | Join target for `fact_sales` — same uniformity reason as every other dimension |
| `order_id` | Business key, traces back to source |
| `customer_id` | Lets you filter "this customer's orders" without going through the fact table |
| `order_date` | Order header date — also independently available via `dim_date` through the fact, kept here too since it's a natural attribute of the order itself |
| `shipping_tier_id`, `supplier_id` | Carried as plain IDs — no backing lookup table exists for either (no `dim_supplier`, no shipping-tier names), so they stay undecoded here rather than needing their own dimension |
| `order_channel` | The one attribute with real reporting value — "Online" vs "Retail PoS" — worth a dimension column on its own |

In [0]:
dim_orders_df = spark.table("gbmart.silver.orders").select(
    sha2(col("order_id"), 256).alias("order_sk"),
    "order_id", "customer_id", "order_date",
    "shipping_tier_id", "supplier_id", "order_channel"
)

dim_orders_df.write.format("delta").mode("overwrite").saveAsTable("gbmart.gold.dim_orders")
print(f"dim_orders rows: {spark.table('gbmart.gold.dim_orders').count():,}")

## Step 7 — Verify All Dimensions

In [0]:
for table in ["dim_customer", "dim_product", "dim_date", "dim_address", "dim_payment_method", "dim_orders"]:
    df = spark.table(f"gbmart.gold.{table}")
    print(f"{table:25s}: {df.count():>8,} rows, {len(df.columns)} columns")

## Reset (if needed)

In [0]:
# for table in ["dim_customer", "dim_product", "dim_date", "dim_address", "dim_payment_method"]:
#     spark.sql(f"DROP TABLE IF EXISTS gbmart.gold.{table}")
# print("Reset complete")